In [1]:
import os, pathlib
import pandas as pd

df = pd.read_csv("./CEAS_08.csv")
# df.head()



# Take a subset of the dataset (20% of each label)
df_label_0 = df[df['label'] == 0].sample(frac=0.001, random_state=42)
df_label_1 = df[df['label'] == 1].sample(frac=0.001, random_state=42)

df = pd.concat([df_label_0, df_label_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"New dataset size: {len(df)}")
print("Distribution of labels in the subset:")
print(df['label'].value_counts())


New dataset size: 39
Distribution of labels in the subset:
label
1    22
0    17
Name: count, dtype: int64


Imports & Setup

In [2]:
from __future__ import annotations
import os
import gc
import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

# TextAttack imports
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import TextFoolerJin2019
from textattack import Attacker
from textattack.datasets import Dataset as TADataset

from datetime import datetime

2025-08-30 01:49:54.558469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756496994.572977 1468475 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756496994.577741 1468475 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756496994.591283 1468475 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756496994.591296 1468475 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756496994.591298 1468475 computation_placer.cc:177] computation placer alr

GPU Sanity Check

In [3]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

Torch version: 2.8.0+cu128
CUDA available: True
Device count: 1
Device name: NVIDIA GeForce RTX 4060 Ti


Log Message Utility

In [4]:
# ----------------------------
# Simple Logging Utility
# ----------------------------
def log_message(save_dir: str, msg: str, console: bool = True):
    log_path = os.path.join(save_dir, "training_log.txt")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}\n"
    with open(log_path, "a") as f:
        f.write(line)
    if console:
        print(line, end="")

Dataset Utilities

In [5]:
@dataclass
class EncodedBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor


class TextClassificationDataset(TorchDataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

albert-based classifier

In [7]:
def train_discriminator(
    train_df: pd.DataFrame,
    save_dir: str = "./albert_phishing_model",
    model_name: str = "albert-base-v2",
    max_length: int = 256,
    train_batch_size: int = 16,
    learning_rate: float = 5e-5,
    num_train_epochs: int = 3,
    gradient_accumulation_steps: int = 1,
    fp16: bool = True,
    weight_decay: float = 0.01,
    logging_steps: int = 50,
    seed: int = 42,
    round_id: Optional[int] = None,
):
    """
    Train a PyTorch ALBERT classifier on the given training set.
    Skips training if a model already exists in `save_dir`.
    Returns: (model, tokenizer, trainer)
    """
    assert {"body", "label"}.issubset(train_df.columns)

    os.makedirs(save_dir, exist_ok=True)
    set_seed(seed)

    round_tag = f"[Round {round_id}]" if round_id else "[Discriminator]"

    # Skip training if model already exists
    model_file = os.path.join(save_dir, "pytorch_model.bin")
    config_file = os.path.join(save_dir, "config.json")
    if os.path.exists(model_file) and os.path.exists(config_file):
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSequenceClassification.from_pretrained(save_dir)
        log_message(save_dir, f"{round_tag} Found existing model in {save_dir}. Skipping training.")
        return model, tokenizer, None

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    num_labels = int(pd.Series(train_df["label"]).nunique())
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    # Dataset
    train_ds = TextClassificationDataset(
        train_df["body"].astype(str).tolist(),
        train_df["label"].astype(int).tolist(),
        tokenizer,
        max_length,
    )

    # Training args (no eval set)
    args = TrainingArguments(
        output_dir=save_dir,
        per_device_train_batch_size=train_batch_size,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        gradient_accumulation_steps=gradient_accumulation_steps,
        fp16=fp16,
        weight_decay=weight_decay,
        logging_steps=logging_steps,
        save_strategy="epoch",
        save_total_limit=2,
        report_to=[],
        logging_dir=os.path.join(save_dir, "runs"),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        tokenizer=tokenizer,
    )

    log_message(save_dir, f"{round_tag} Starting training...")
    trainer.train()

    model_save_dir = os.path.join(save_dir, "model")
    trainer.save_model(model_save_dir)
    tokenizer.save_pretrained(model_save_dir)
    log_message(model_save_dir, f"{round_tag} Training completed. Model saved to {model_save_dir}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model, tokenizer, trainer

adversarial sample generation (TextAttack)

In [8]:
# ----------------------------
# Adversarial Generation (TextAttack)
# ----------------------------

from tqdm import tqdm

def generate_adversarial_samples(
    model: torch.nn.Module,
    tokenizer,
    dataset_df: pd.DataFrame,
    recipe: str = "textfooler",
    max_examples: Optional[int] = None,
    seed: int = 42,
) -> Tuple[pd.DataFrame, dict]:
    """Generate adversarial samples using TextAttack with a live progress bar."""
    from textattack.models.wrappers import HuggingFaceModelWrapper
    from textattack.datasets import Dataset as TADataset
    from textattack import Attacker
    from textattack.attack_recipes import TextFoolerJin2019, PWWSRen2019, DeepWordBugGao2018

    assert {"body", "label"}.issubset(dataset_df.columns)

    set_seed(seed)
    tuples = [(str(row["body"]), int(row["label"])) for _, row in dataset_df.iterrows()]
    if max_examples is not None:
        tuples = tuples[: max(0, int(max_examples))]

    ta_dataset = TADataset(tuples)

    if torch.cuda.is_available():
        model.to("cuda")
    wrapper = HuggingFaceModelWrapper(model, tokenizer)
    if torch.cuda.is_available():
        wrapper.model.to("cuda")   # ensure TextAttack wrapper uses GPU

    if recipe == "textfooler":
        attack = TextFoolerJin2019.build(wrapper)
    elif recipe == "pwws":
        attack = PWWSRen2019.build(wrapper)
    elif recipe == "deepwordbug":
        attack = DeepWordBugGao2018.build(wrapper)
    else:
        raise ValueError(f"Unsupported recipe: {recipe}")

    attacker = Attacker(attack, ta_dataset)

    adv_rows = []
    success, fail, skipped = 0, 0, 0

    print(f"Starting adversarial attack with {recipe} on {len(ta_dataset)} samples...")
    for result in tqdm(attacker.attack_dataset(), total=len(ta_dataset), ncols=100, desc=f"{recipe} attack"):
        if getattr(result, "perturbed_result", None) is not None:
            adv_text = result.perturbed_result.attacked_text.text
            original_label = int(result.original_result.ground_truth_output)
            adv_rows.append({"body": adv_text, "label": original_label})
            success += 1
        else:
            if result.goal_function_result.succeeded:
                skipped += 1
            else:
                fail += 1

    adv_df = pd.DataFrame(adv_rows, columns=["body", "label"]).dropna()
    stats = {"success": success, "fail": fail, "skipped": skipped}

    print(f"Finished {recipe} attack: {success} success, {fail} fail, {skipped} skipped.\n")
    return adv_df, stats

multi-round adversarial training game

In [9]:
# ----------------------------
# Adversarial Training Game Loop
# ----------------------------

def adversarial_training_game(
    df: pd.DataFrame, #train pool only-->80%
    rounds: int = 5,
    save_dir: str = "./albert_game_model",
    adv_samples_per_round: Optional[int] = None,  # default = full dataset
    model_name: str = "albert-base-v2",
    max_length: int = 256,
    train_batch_size: int = 16,
    learning_rate: float = 5e-5,
    num_train_epochs: int = 3,
    gradient_accumulation_steps: int = 1,
    fp16: bool = True,
    weight_decay: float = 0.01,
    logging_steps: int = 50,
    seed: int = 42,
    resume_round: int = 1,
):
    os.makedirs(save_dir, exist_ok=True)
    assert {"body", "label"}.issubset(df.columns)
    df = df.copy()
    recipes = ["textfooler", "pwws", "deepwordbug"]

    if adv_samples_per_round is None:
        adv_samples_per_round = len(df)  # full train dataset

    # Initialize tokenizer and model once
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_path = os.path.join(save_dir, "base_model")
    if resume_round == 1 or not os.path.exists(model_path):
        num_labels = int(pd.Series(df["label"].tolist()).nunique())
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        print(f"Initialized new ALBERT model with {num_labels} labels.")
    else:
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        print(f"Loaded existing base model from {model_path}.")

    # Main adversarial training loop
    for r in range(resume_round, rounds + 1):
        print(f"\n=== START ROUND {r} / {rounds} ===")
        print(f"Current dataset size: {len(df)}")
        log_message(save_dir, f"===== Round {r} / {rounds} =====")

        # Checkpoint directory for this round
        round_dir = os.path.join(save_dir, f"round_{r}")
        os.makedirs(round_dir, exist_ok=True)

        # Merge previous adversarial examples if resuming
        if r > 1:
            prev_adv_path = os.path.join(save_dir, f"adversarial_round_{r-1}.csv")
            if os.path.exists(prev_adv_path):
                adv_prev = pd.read_csv(prev_adv_path)
                if not adv_prev.empty:
                    before = len(df)
                    df = pd.concat([df, adv_prev], ignore_index=True)
                    df.drop_duplicates(subset=["body"], inplace=True)
                    after = len(df)
                    print(f"Merged previous adversarial samples: dataset size {before} -> {after}")
                    log_message(save_dir, f"Resuming round {r}: dataset size {before} -> {after}")

        model, tokenizer, trainer = train_discriminator(
          train_df=df,                # renamed arg
          save_dir=round_dir,
          model_name=model_name,
          max_length=max_length,
          train_batch_size=train_batch_size,
          learning_rate=learning_rate,
          num_train_epochs=num_train_epochs,
          gradient_accumulation_steps=gradient_accumulation_steps,
          fp16=fp16,
          weight_decay=weight_decay,
          logging_steps=logging_steps,
          seed=seed,
          round_id=r,
      )

        # Save base model after first training round
        if r == 1:
            base_model_dir = os.path.join(save_dir, "base_model")
            model.save_pretrained(base_model_dir)
            tokenizer.save_pretrained(base_model_dir)
            print(f"Saved base model after round 1 to {base_model_dir}")

        # Generate adversarial examples
        base_for_attack = df.sample(n=min(adv_samples_per_round*2, len(df)), random_state=seed).reset_index(drop=True)
        combined_adv = []
        for recipe in recipes:
          # Paths for full texts and stats
          adv_texts_path = os.path.join(round_dir, f"round_{r}_{recipe}_adversarial_texts.csv")
          stats_path = os.path.join(round_dir, f"round_{r}_{recipe}_stats.csv")

          # Check if the adversarial texts file already exists
          if os.path.exists(adv_texts_path):
              # Load previously saved results
              adv_df = pd.read_csv(adv_texts_path)
              stats_df = pd.read_csv(stats_path)
              stats = stats_df.to_dict(orient="records")[0]  # convert one-row DataFrame to dict
              print(f"Loaded existing results for {recipe} from {adv_texts_path}")
          else:
              # Run attack if file does not exist
              adv_df, stats = generate_adversarial_samples(
                  model=model,
                  tokenizer=tokenizer,
                  dataset_df=base_for_attack,
                  max_examples=adv_samples_per_round,
                  recipe=recipe,
                  seed=seed,
              )
              # Save full adversarial texts
              adv_df.to_csv(adv_texts_path, index=False)
              print(f"Saved {len(adv_df)} successful adversarial samples to {adv_texts_path}")

              # Save just the stats
              stats_df = pd.DataFrame([stats])  # convert dict to one-row DataFrame
              stats_df.to_csv(stats_path, index=False)
              print(f"Saved stats to {stats_path}")

          # Keep track of combined adversarial examples
          if not adv_df.empty:
              combined_adv.append(adv_df)


        adv_df = pd.concat(combined_adv, ignore_index=True) if combined_adv else pd.DataFrame(columns=["body", "label"])
        print(f"Total adversarial samples collected this round: {len(adv_df)}")
        log_message(save_dir, f"Total adversarial samples collected this round: {len(adv_df)}")

        # Merge & dedupe dataset for next round
        if not adv_df.empty:
            before = len(df)
            df = pd.concat([df, adv_df], ignore_index=True)
            df.drop_duplicates(subset=["body"], inplace=True)
            after = len(df)
            print(f"Dataset size after merging adversarial samples: {before} -> {after}")
            log_message(save_dir, f"Dataset size after merge: {before} -> {after}")

        # Save combined adversarial set
        adv_path = os.path.join(save_dir, f"round_{r}_augmented_dataset.csv")
        adv_df.to_csv(adv_path, index=False)
        print(f"Saved combined adversarial set to {adv_path}")
        log_message(save_dir, f"Saved combined adversarial samples to {adv_path}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"=== END ROUND {r} ===\n")

    return model, tokenizer, df

In [10]:
import nltk
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")   # also needed for synonyms
nltk.download("omw-1.4")   # WordNet dependencies

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/nazmul/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/nazmul/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/nazmul/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Begin Training GAME

In [11]:
from sklearn.model_selection import train_test_split

# Split once into train_pool (80%) and dev_set (20%)
train_pool, dev_set = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

dev_set.to_csv("validation_set_0.csv", index=False)
print(f"Saved dev set to validation_set_0.csv ({len(dev_set)} samples)")

rounds = 3

Saved dev set to validation_set_0.csv (8 samples)


In [12]:
final_model, final_tokenizer, final_dataset = adversarial_training_game(
        df=train_pool,
        rounds=rounds,
        save_dir="./albert_game_modelv4",
        model_name="albert-base-v2",
        max_length=256,
        train_batch_size=8,
        learning_rate=5e-5,
        num_train_epochs=3,
        gradient_accumulation_steps=1,
        fp16=True,
        weight_decay=0.01,
        logging_steps=50,
        seed=42,
)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initialized new ALBERT model with 2 labels.

=== START ROUND 1 / 3 ===
Current dataset size: 31
[2025-08-30 01:49:59] ===== Round 1 / 3 =====


Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1468475/2263762156.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


[2025-08-30 01:50:01] [Round 1] Starting training...


Step,Training Loss


...

--------------------------------------------- Result 9 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

AWhat we offer is a erliable way of maculine pwoer enhancement!
Do'nt wit anymore to impldement these changes!
http://placefall.com/


Boom/bust sequences are pronecannot opssibly discoutnSometimes, the divergence was large and not self-correcting. This




[Succeeded / Failed / Skipped / Total] 0 / 10 / 0 / 10: 100%|██████████| 10/10 [02:11<00:00, 13.17s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

      
          Find your love stick gain here
        CLICK HERE URl!!!!
        qOqZCJyF9855v2
        
    
   



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 0      |
| Number of failed attacks:     | 10     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 100.0% |
| Attack success rate:          | 0.0%   |
| Average perturbed word %:     | nan%   |
| Average num. words per input: | 160.5  |
| Avg num queries:              | 639.9  |
+-------------------------------+--------+

deepwordbug attack:  32%|████████████▌                          | 10/31 [00:00<00:00, 134432.82it/s]


Finished deepwordbug attack: 10 success, 0 fail, 0 skipped.

Saved 10 successful adversarial samples to ./albert_game_model/round_3/round_3_deepwordbug_adversarial_texts.csv
Saved stats to ./albert_game_model/round_3/round_3_deepwordbug_stats.csv
Total adversarial samples collected this round: 30
[2025-08-30 02:14:55] Total adversarial samples collected this round: 30
Dataset size after merging adversarial samples: 110 -> 140
[2025-08-30 02:14:55] Dataset size after merge: 110 -> 140
Saved combined adversarial set to ./albert_game_model/round_3_augmented_dataset.csv
[2025-08-30 02:14:55] Saved combined adversarial samples to ./albert_game_model/round_3_augmented_dataset.csv
=== END ROUND 3 ===



In [13]:
save_dir = "./adversarial_val_sets"
os.makedirs(save_dir, exist_ok=True)

recipes = {
    "textfooler": TextFoolerJin2019,
    "pwws": PWWSRen2019,
    "deepwordbug": DeepWordBugGao2018,
}

for r in range(1, rounds + 1):
    model_dir = os.path.join("albert_game_model", f"round_{r}", "model")
    assert os.path.exists(model_dir), f"Model dir not found: {model_dir}"

    print(f"\n====================== Round {r} ======================\n")

    # Load discriminator for this round
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    if torch.cuda.is_available():
        model.to("cuda")

    adv_dfs = []

    for recipe_name, recipe_cls in recipes.items():
        print(f"--> Running {recipe_name} attack on dev set")

        tuples = [(str(row["body"]), int(row["label"])) for _, row in dev_set.iterrows()]
        ta_dataset = TADataset(tuples)

        wrapper = HuggingFaceModelWrapper(model, tokenizer)
        if torch.cuda.is_available():
            wrapper.model.to("cuda")

        attack = recipe_cls.build(wrapper)
        attacker = Attacker(attack, ta_dataset)

        adv_rows = []
        success, fail, skipped = 0, 0, 0

        for result in tqdm(attacker.attack_dataset(), total=len(ta_dataset), ncols=100, desc=f"{recipe_name}"):
            if getattr(result, "perturbed_result", None) is not None:
                adv_text = result.perturbed_result.attacked_text.text
                original_label = int(result.original_result.ground_truth_output)
                adv_rows.append({"body": adv_text, "label": original_label})
                success += 1
            else:
                if result.goal_function_result.succeeded:
                    skipped += 1
                else:
                    fail += 1

        adv_df = pd.DataFrame(adv_rows, columns=["body", "label"]).dropna()
        adv_dfs.append(adv_df)

        print(f"[Round {r}][{recipe_name}] success={success}, fail={fail}, skipped={skipped}")

    # Merge clean dev + all adversarial samples
    combined_df = pd.concat([dev_df] + adv_dfs, ignore_index=True)

    out_path = os.path.join(save_dir, f"augmented_validation_set_{r}.csv")
    combined_df.to_csv(out_path, index=False)

    print(f"\n[Round {r}] Saved augmented validation set with all recipes to {out_path}\n")

print("Finished generating adversarial validation sets for all rounds and recipes.")

NameError: name 'PWWSRen2019' is not defined